# 单比特最低 20 个能级的衰减率及 $T_1/T_\phi$

复现 `cz/scripts/run_2q_fidelity.py` 的裸单比特噪声模型，分别计算参考时间 **170、30、3 μs**。

- NPZ 中的 `n_theta1/2` 是原始矩阵，不需要再除以 $2\pi$。衰减率由矩阵元和参考跃迁 $8\to4$ 推导，并非直接存储在文件中。
- $\Gamma_{j\to i}=|n_{ij}|^2/[1000\,T_{\rm ref}|n_{48}|^2]$（ns⁻¹），$T_{1,j}=1/\sum_{i<j}\Gamma_{j\to i}$。
- `full` 包含所有向下通道；`filtered` 使用 fidelity 脚本的 charge basis 和阈值。
- `Tphi_mapped_us` 按物理能级索引；`Tphi_current_us` 复现当前脚本的连续索引，便于检查错位。
- 独立投影噪声 $L_j=\sqrt{2\gamma_j}|j\rangle\langle j|$ 下，$T_\phi$ 指 $\rho_{0j}$ 的纯退相干时间。一般 $\rho_{ij}$ 的纯退相干率为 $\gamma_i+\gamma_j$。
- `inf` 表示模型中的零速率；`nan` 表示未包含在 charge basis 中。这里只计算裸单比特，不包含两比特缀饰或驱动效应。


In [1]:
from pathlib import Path
import csv
import html
import numpy as np
from IPython.display import display, HTML

# ==================== 集中配置区 ====================
# 支持从项目根目录、cz/ 或更深的 notebook 目录启动。
PROJECT_DIR = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "ham_data.py").is_file() and (p / "cz").is_dir()
)
SYSTEM_PATH = PROJECT_DIR / "data/Two_qubit_data_Sorted_Truc/two_qubit_data.npz"
DEPHASE_PATH = PROJECT_DIR / "data/flux_derivative_truc500/gamma_phi_2Q_truc500.npz"
OUTPUT_DIR = PROJECT_DIR / "cz/data/single_qubit_lifetimes"
REFERENCE_TIMES_US = [170.0, 30.0, 3.0]
N_LEVELS = 20
FILTER_RATIO = 0.3
LOW_STATES = 50
DECAY_ENLARGE = 1.0  # 跳跃算符振幅放大倍数，速率乘其平方。
SAVE_CSV = True


In [2]:
# ==================== 数据加载区 ====================
# 仅提取单比特矩阵和 charge basis，不读取巨大的两比特本征矢。
with np.load(SYSTEM_PATH, allow_pickle=True) as archive:
    data = {key: archive[key] for key in (
        "n_theta1", "n_theta2", "hspace_1_charge", "hspace_2_charge"
    )}
with np.load(DEPHASE_PATH) as archive:
    noise = {key: archive[key] for key in ("q0", "q1")}
print("System:", SYSTEM_PATH)
print("Dephasing:", DEPHASE_PATH)
for key, value in {**data, **noise}.items():
    print(key, value.shape)


System: /home/zlqed/two_qubit_gate/zeropi/two_qubit_gate_clean/data/Two_qubit_data_Sorted_Truc/two_qubit_data.npz
Dephasing: /home/zlqed/two_qubit_gate/zeropi/two_qubit_gate_clean/data/flux_derivative_truc500/gamma_phi_2Q_truc500.npz
n_theta1 (300, 300)
n_theta2 (300, 300)
hspace_1_charge (154,)
hspace_2_charge (155,)
q0 (500,)
q1 (500,)


In [3]:
# ==================== 噪声模型与计算区 ====================
def inverse_rate_us(rate):
    """Convert ns^-1 to microseconds; zero rate means infinite lifetime."""
    return np.inf if rate == 0 else 1.0 / (1000.0 * rate)


def calculate_tables(reference_us):
    levels, transitions = [], []
    for q in range(2):
        matrix = data[f'n_theta{q + 1}']  # RAW matrix, no 2*pi.
        hspace = data[f'hspace_{q + 1}_charge'].astype(int)
        if np.any(np.diff(hspace) <= 0):
            raise ValueError('Expected increasing charge-selected level indices.')
        reference = abs(matrix[4, 8]) ** 2
        if reference == 0:
            raise ValueError('Zero reference matrix element for 8 -> 4.')
        prefactor = 1.0 / (1000 * reference_us * reference)
        low = hspace[hspace < LOW_STATES]
        n_max = np.abs(matrix[np.ix_(low, low)]).max()
        selected_position = {int(s): i for i, s in enumerate(hspace)}
        gamma = np.abs(noise[f'q{q}'] / reference_us)
        for j in range(min(N_LEVELS, len(matrix))):
            rates = prefactor * np.abs(matrix[:j, j]) ** 2 * DECAY_ENLARGE**2
            full_rate = float(rates.sum())
            included = j in selected_position
            cutoff = n_max * FILTER_RATIO * (1 if j < LOW_STATES else 10)
            filtered_rate = 0.0
            for i, rate in enumerate(rates):
                retained = included and i in selected_position and abs(matrix[i, j]) > cutoff
                if retained:
                    filtered_rate += float(rate)
                transitions.append(dict(qubit=q, source=j, target=i,
                                        rate_per_ns=float(rate), retained=retained))
            # Ground projector is omitted by construct_c_ops_2q. For an
            # independent sqrt(2*gamma_j)|j><j| channel, rho_0j decays
            # at gamma_j; rho_ij (i,j>0) at gamma_i + gamma_j.
            mapped_gamma = float(gamma[j]) if j else 0.0
            actual_gamma = (float(gamma[selected_position[j]]) if included and j else
                            (0.0 if j == 0 else np.nan))
            levels.append(dict(
                qubit=q, level=j, in_charge_basis=included,
                charge_position=selected_position.get(j, np.nan),
                decay_full_per_ns=full_rate,
                T1_full_us=inverse_rate_us(full_rate),
                decay_filtered_per_ns=filtered_rate if included else np.nan,
                T1_filtered_us=inverse_rate_us(filtered_rate) if included else np.nan,
                gamma_phi_mapped_per_ns=mapped_gamma,
                Tphi_mapped_us=inverse_rate_us(mapped_gamma),
                gamma_phi_current_per_ns=actual_gamma,
                Tphi_current_us=inverse_rate_us(actual_gamma),
            ))
    return levels, transitions



In [4]:
# 每组参考时间均计算两个单比特的 level 0--19。
all_levels, all_transitions = [], []
for reference_us in REFERENCE_TIMES_US:
    levels, transitions = calculate_tables(reference_us)
    all_levels.extend(dict(reference_us=reference_us, **row) for row in levels)
    all_transitions.extend(dict(reference_us=reference_us, **row) for row in transitions)

columns = ["level", "decay_full_per_ns", "T1_full_us", "T1_filtered_us",
           "Tphi_mapped_us", "Tphi_current_us"]
for reference_us in REFERENCE_TIMES_US:
    for qubit in range(2):
        rows = [r for r in all_levels if r["reference_us"] == reference_us and r["qubit"] == qubit]
        print(f"Reference = {reference_us:g} μs, qubit {qubit}")
        header = "".join(f"<th>{html.escape(c)}</th>" for c in columns)
        body = "".join("<tr>" + "".join(f"<td>{r[c]:.7g}</td>" for c in columns) + "</tr>" for r in rows)
        display(HTML(f"<table><tr>{header}</tr>{body}</table>"))

if SAVE_CSV:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name, rows in [("levels_170_30_3us", all_levels), ("transitions_170_30_3us", all_transitions)]:
        path = OUTPUT_DIR / f"{name}.csv"
        with path.open("w", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
        print("Saved:", path)


Reference = 170 μs, qubit 0


level,decay_full_per_ns,T1_full_us,T1_filtered_us,Tphi_mapped_us,Tphi_current_us
0,0,inf,inf,inf,inf
1,2.259149e-06,442.6445,442.6445,2.414987e+07,2.414987e+07
2,1.293764e-10,7729383,inf,170,170
3,2.048404e-30,4.881849e+26,nan,169.998,nan
4,4.274229e-06,233.9603,233.9603,6.178831e+07,169.998
5,2.225095e-06,449.419,454.3985,1112.063,6.178831e+07
6,2.207579e-06,452.9849,nan,1112.073,nan
7,2.106309e-09,474764.1,nan,7.450353e+07,nan
8,5.889768e-06,169.786,170,1.39728e+07,1112.063
9,4.571513e-06,218.7459,272.3855,11844.13,1112.073


Reference = 170 μs, qubit 1


level,decay_full_per_ns,T1_full_us,T1_filtered_us,Tphi_mapped_us,Tphi_current_us
0,0,inf,inf,inf,inf
1,2.322218e-06,430.6229,430.6229,3.76103e+07,3.76103e+07
2,4.613609e-10,2167500,inf,170,170
3,2.043949e-31,4.892491e+27,nan,169.9937,nan
4,4.367651e-06,228.956,228.956,8753167,169.9937
5,2.307589e-06,433.3528,448.1733,1279.075,8753167
6,2.252579e-06,443.9356,nan,1279.36,nan
7,5.518957e-09,181193.7,nan,1.827396e+08,nan
8,5.902659e-06,169.4152,170,7003421,1279.075
9,5.191396e-06,192.6264,192.9461,12572.84,1279.36


Reference = 30 μs, qubit 0


level,decay_full_per_ns,T1_full_us,T1_filtered_us,Tphi_mapped_us,Tphi_current_us
0,0,inf,inf,inf,inf
1,1.280185e-05,78.11374,78.11374,4261742,4261742
2,7.331332e-10,1364009,inf,30,30
3,1.160762e-29,8.615028e+25,nan,29.99965,nan
4,2.422063e-05,41.28711,41.28711,1.090382e+07,29.99965
5,1.260887e-05,79.30924,80.18798,196.2464,1.090382e+07
6,1.250961e-05,79.93852,nan,196.2481,nan
7,1.193575e-08,83781.89,nan,1.314768e+07,nan
8,3.337535e-05,29.96223,30,2465789,196.2464
9,2.590524e-05,38.60223,48.06803,2090.141,196.2481


... (3 outputs left out)

level,decay_full_per_ns,T1_full_us,T1_filtered_us,Tphi_mapped_us,Tphi_current_us
0,0,inf,inf,inf,inf
1,0.0001280185,7.811374,7.811374,426174.2,426174.2
2,7.331332e-09,136400.9,inf,3,3
3,1.160762e-28,8.615028e+24,nan,2.999965,nan
4,0.0002422063,4.128711,4.128711,1090382,2.999965
5,0.0001260887,7.930924,8.018798,19.62464,1090382
6,0.0001250961,7.993852,nan,19.62481,nan
7,1.193575e-07,8378.189,nan,1314768,nan
8,0.0003337535,2.996223,3,246578.9,19.62464
9,0.0002590524,3.860223,4.806803,209.0141,19.62481


Reference = 3 μs, qubit 1


level,decay_full_per_ns,T1_full_us,T1_filtered_us,Tphi_mapped_us,Tphi_current_us
0,0,inf,inf,inf,inf
1,0.0001315923,7.599227,7.599227,663711.2,663711.2
2,2.614379e-08,38250.01,inf,3,3
3,1.158238e-29,8.633807e+25,nan,2.999888,nan
4,0.0002475002,4.0404,4.0404,154467.7,2.999888
5,0.0001307634,7.647403,7.908941,22.57191,154467.7
6,0.0001276461,7.834158,nan,22.57693,nan
7,3.127409e-07,3197.535,nan,3224816,nan
8,0.000334484,2.989679,3,123589.8,22.57191
9,0.0002941791,3.39929,3.404931,221.8737,22.57693


Saved: /home/zlqed/two_qubit_gate/zeropi/two_qubit_gate_clean/cz/data/single_qubit_lifetimes/levels_170_30_3us.csv
Saved: /home/zlqed/two_qubit_gate/zeropi/two_qubit_gate_clean/cz/data/single_qubit_lifetimes/transitions_170_30_3us.csv
